# AI Integrations for Developers — Exam

## Instructions

- This notebook is a **template** where you must put your code.  
- You should **fill in all empty variables** and complete the code so that when I download your notebook and click **Run all**, all cells execute correctly and provide the answers.  
- ⚠️ **Do NOT hardcode your API key**. Use Colab environment variables (`%env OPENAI_API_KEY=your_key_here`) and access them in your code.  
- You may **create more cells** if needed. It is recommended that your code is well-structured and split logically into separate cells.  
- The function **`ask_ai(query)`** must be implemented by you. All queries will call this function to check your solution.  
- ✅ **Test cases will be created by me (the instructor).** You are **not allowed to modify, remove, or add to the test cases cell**. Your code must work correctly with the provided test cases.  
- You are **ONLY ALLOWED** to use only the following:  
  - **Models:** OpenAI or Anthropic  
  - **Technologies:** LangChain or vanilla Python code  
  - **Vector Store:** Chroma DB

🚨 **Any student who does not follow the template, does not stick to the required format, or whose code does not execute properly will be disqualified.**


### Important

Fill in **all the variables** in the cell.  
❌ **Do NOT put your API key directly in the code.**  
✅ The cell must be set up to take the API key from the Colab environment variables.


In [136]:
# ================================
# 🔧 RAG Configuration Variables
# ================================

# ⚠️ Do NOT put your API key here directly.
# Make sure you set your API key in Colab like this:
# %env OPENAI_API_KEY=your_key_here

import os
from google.colab import userdata

# API Key (taken from Colab environment variables)
API_KEY = userdata.get("OPENAI_API_KEY")

os.environ["OPENAI_API_KEY"] = API_KEY

# Prompt & Model Settings:

# 1. Generates PDF summary
SYSTEM_MESSAGE_PDF_SUMMARY_GENERATOR = """
<context>
We are building a chatbot to answer users' queries based on information from a PDF file. The PDF file is chunked and stored in a vector database. We need a concise summary that will help an AI assistant optimize user queries for better vectorstore retrieval. The summary will be used in a query optimization step where an AI assistant receives both this summary and a user's original query to generate an optimized search query for the vectorstore.
</context>

<role>
You are an experienced Knowledge Manager specializing in document analysis for RAG systems.

<skills>
- Extract key topics, themes, and subject areas from documents
- Identify important terminology and domain-specific vocabulary
- Create concise overviews that capture document scope without detail
- Structure information for AI query optimization
- Distinguish between high-level concepts and specific details
- Understand how document summaries aid in search query formulation
</skills>

<experience>
- 3+ years working with retrieval-augmented generation systems or AI-powered search
- Background in prompt engineering or AI system optimization
- Knowledge of how document structure affects AI retrieval performance
- Experience preparing content for vector databases and understanding chunking strategies
- Experience creating executive summaries, abstracts, or document overviews
- Understanding of how users formulate queries and search for information
</experience>
</role>

<task>
Create a brief summary of the EXTRACTED TEXT FROM PDF FILE that includes:
1. Main Topics: What subjects/areas does the document cover?
2. Key Terminology: Important terms, concepts, or vocabulary used
3. Document Structure: Major sections or categories of information
4. Content Types: What kinds of information can users expect to find (procedures, data, policies, numbers, etc.)

<important>
- Focus on WHAT the EXTRACTED TEXT FROM PDF FILE contains, not the specific details
- Use terminology from the original EXTRACTED TEXT FROM PDF FILE
- Keep it concise
- Structure it to help with query optimization
</important>
</task>

<next>
Output only the summary text without additional formatting or commentary.
</next>
"""

HUMAN_MESSAGE_PDF_SUMMARY_GENERATOR = """
EXTRACTED TEXT FROM PDF FILE: {extracted_text}
"""

# 2. Contextualizes Chunks using the PDF summary
SYSTEM_MESSAGE_CHUNK_CONTEXTUALIZER = """
<role>
You are a document processing specialist. Your task is to add contextual information to document chunks to improve their retrieval in a vector database.
</role>

<task>
Given a document CHUNK and a SUMMARY of the source document, provide a brief contextual prefix (50-100 tokens) that situates the CHUNK within the overall document. This context should:
- Identify which section or topic the CHUNK relates to based on the document SUMMARY
- Clarify any references that might be unclear when the CHUNK stands alone
- Preserve the meaning and searchability of the original SUMMARY
- Use terminology from the document  SUMMARY and CHUNK
<task>

<next>
Output only the contextual prefix, nothing else.
</next>
"""

HUMAN_MESSAGE_CHUNK_CONTEXTUALIZER = """
SUMMARY:\n{text_summary}\n\n
CHUNK:\n{chunk_content}
"""

# 3. Creates optimized query using the PDF summary
SYSTEM_MESSAGE_QUERY_OPTIMIZER = """
<context>
Our client hired us to develop a chatbot that answers user questions based on information from their PDF document. The PDF has been processed, chunked, and stored in a vectorstore. Users interact with the chatbot through conversational questions, and the system's effectiveness depends entirely on successfully retrieving the most relevant chunks from the vectorstore. Query optimization is critical because poor retrieval leads to irrelevant or incomplete responses, directly impacting user satisfaction and system performance.
</context>

<role>
You are an Information Architect specializing in query optimization for RAG (Retrieval-Augmented Generation) systems. Your task is to analyze user queries and transform them into optimized search queries that will retrieve the most relevant content from a vectorstore.

<skills>
- Parse user intent from conversational queries
- Identify key concepts, entities, and relationships in unstructured text
- Handle ambiguous, incomplete, or poorly structured user inputs
- Recognize synonyms and related terms that might exist in the vectorstore
- Transform natural language into effective search terms
- Understand vector similarity matching and semantic search principles
- Balance query specificity vs. breadth for optimal retrieval
- Map user concepts to document-specific vocabulary and terminology
- Recognize hierarchical relationships and implicit requirements
- Ensure optimized queries align with vectorstore chunking structure
- Maintain user intent while maximizing retrieval success
</skills>

<experience>
- 3+ years working with search engines, vector databases, or recommendation systems
- Experience with semantic search, embeddings, and similarity matching
- Knowledge of retrieval metrics (precision, recall, relevance scoring)
- Understanding of how document chunking affects search performance
- Experience building or optimizing retrieval-augmented generation systems
- Understanding of how retrieval quality impacts downstream generation
- Knowledge of prompt engineering and context window optimization
- Experience with vector stores (Pinecone, Weaviate, Chroma, etc.)
- Experience building AI-powered applications for business users
- Understanding of how users search for and consume information
</experience>
</role>

<task>
1. Analazy the USER QUERY
1. Study the provided PDF SUMMARY to understand: (1) main topics covered, (2) key terminology used, (3) document structure, and (4) types of information available. This knowledge will guide your query optimization decisions
2. Create one optimized search query that will effectively retrieve relevant content from the vectorstore

<important>
- Focus on terms, concepts, and phrases that are likely to match the chunked content while maintaining the user's original intent expressed into the USER QUERY
- Your optimized query should use terminology and concepts present in the source document to ensure successful retrieval from the vectorstore
- Do not include irrelevant keywords that could block matching the user’s intended information
</important>
</task>

<next>
Output only the optimized search query as a question in the customer's voice. Do not include anything else except the question.
</next>
"""

HUMAN_MESSAGE_QUERY_OPTIMIZER = """
USER QUERY:\n{query}\n\n
PDF SUMMARY:\n{text_summary}
"""

# 4. Responds to the query
SYSTEM_MESSAGE_RESPONDER = """
<context>
Our client hired us to develop a chatbot that answers user questions based on information from their PDF document.
</context>

<role>
You are a customer support representative for a PDF-based Q&A chatbot system.

<skills>
- Outstanding written communication skills with ability to explain complex information clearly
- Active listening skills to understand the true intent behind user questions
- Strong analytical skills to interpret ambiguous or incomplete questions
- Ability to make connections between user queries and relevant document sections
- Meticulous accuracy when citing or referencing document information
- Careful verification of information before providing responses
</skills>

<expirience>
- 3+ years in customer support
- Experience handling technical inquiries or document-based support
- Track record of maintaining high customer satisfaction scores
- Background in roles requiring frequent document consultation
- Experience with FAQ maintenance or knowledge base management
</expirience>
</role>

<who_am_I>
- I am a busy professional who need to quickly extract specific information from lengthy documents
- I am most comfortable with conversational interfaces rather than complex search systems
- I prefer natural language queries over technical search syntax
- I want immediate, accurate answers without having to search manually
- I may become frustrated if answers are too vague or don't directly address my needs
- I want confident, authoritative responses backed by document citations
- I value transparency when information isn't available in the document
</who_am_I>

<behaviour>
1. Only provide answers based on information explicitly contained in the provided PDF content
2. When users ask questions outside the document scope, politely redirect them back to document-related topics
3. Maintain and reference information shared during the current conversation (names, preferences, previous questions)
4. Answer basic conversational queries that help maintain rapport and context
5. Examples of acceptable non-document responses:
- "Yes, [User's Name], I remember you asked about that earlier"
- "As you mentioned, you're looking for information about [topic from conversation]"
- "I recall you said your name is [User's Name]"
6. Always end with a complete sentence without cutting off mid-thought, mid-sentence or mid-paragraph.
</behaviour>

<next>
Output only your response to the question.
</next>
"""

HUMAN_MESSAGE_RESPONDER = """
CONVERSATION MEMORY:\n{conversation_memory}\n\n
KEY INFORMATION INCLUDED INTO THE PDF DOCUMENT:\n{text_summary}\n\n
Based on the following CONTEXT:\n{context}\n\n
Respond to the following QUESTION:\n{query}
"""

MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"

# Chunking Parameters
CHUNK_SIZE = 770
CHUNK_OVERLAP = 70
TOP_N_RESULTS = 3

# Generation Parameters
OUTPUT_LENGTH = 500
TEMPERATURE = 0.1
TOP_P = 0.1
FREQUENCY_PENALTY = 1.0
PRESENCE_PENALTY = 1.0

### Code Organization

Create more cells if needed and put your code in them.  
It is **recommended** that your code is well-structured, split logically, and kept in separate cells for clarity.


### 1. Install Dependencies

In [137]:
!pip install -U langchain langchain-openai langchain-community chromadb pypdf python-dotenv

### 2. Upload PDF to Colab

In [138]:
from google.colab import files

uploaded_file = files.upload()

Saving Exam_Preparation_PDF.pdf to Exam_Preparation_PDF (3).pdf


### 3. Load the PDF

In [139]:
from langchain_community.document_loaders import PyPDFLoader

pdf_path = list(uploaded_file.keys())[0]
loader = PyPDFLoader(pdf_path)

pages = loader.load()

### 4. Extract text from PDF

In [140]:
# Extract and concatenate text from all pages
extracted_text = " ".join([page.page_content for page in pages[:5]])
extracted_text = extracted_text.replace('\n', ' ').replace('\n\n', ' ').replace('\r', ' ')
# Clean up extra whitespace
extracted_text = " ".join(extracted_text.split())

### 8. Define function to generate AI responses

In [141]:
from langchain.prompts import ChatPromptTemplate
from langchain.prompts import (
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
)

def generate_ai_response(
    llm,
    system_message,
    human_message,
    **kwargs
):
    prompt = ChatPromptTemplate.from_messages([
        SystemMessagePromptTemplate.from_template(
            system_message
        ),
        HumanMessagePromptTemplate.from_template(
            human_message
        ),
    ])

    messages = prompt.format_messages(**kwargs)

    return llm.invoke(messages).content

### 7. Initialize the Chat Model

In [142]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model=MODEL,
    max_tokens=OUTPUT_LENGTH,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    frequency_penalty=FREQUENCY_PENALTY,
    presence_penalty=PRESENCE_PENALTY,
    streaming=True,
)

### 9. Generate summary of the extracted text from PDF

In [143]:
text_summary = generate_ai_response(
    llm,
    SYSTEM_MESSAGE_PDF_SUMMARY_GENERATOR,
    HUMAN_MESSAGE_PDF_SUMMARY_GENERATOR,
    extracted_text=extracted_text,
  )

### 4. Chunk the extracted text from PDF

#### 4.1. Find the best split point within max_size using semantic separators

In [144]:
import re

def find_best_split_point(emaining_text, max_size):
    # Define separators in order of preference (strongest to weakest semantic boundaries)
    separators = ['. ', '! ', '? ', '; ', ': ', ', ', ' - ', '-', ' (', ' ']

    if len(emaining_text) <= max_size:
        return len(emaining_text)

    # Try each separator in order of preference
    for separator in separators:
        search_text = emaining_text[:max_size + len(separator)]
        last_occurrence = search_text.rfind(separator)

        if last_occurrence != -1:
            # Found a good split point
            return last_occurrence + len(separator)

    # This should never happen with normal text since ' ' is our last separator
    # But included as a safety fallback for edge cases like URLs or encoded data
    return max_size

#### 4.2. Extract overlap text from the end of a chunk, preferring complete sentences/phrases

In [145]:
def extract_overlap_text(text, overlap_size):
    if len(text) <= overlap_size:
        return text

    # Start from the desired overlap position and work backwards
    start_pos = len(text) - overlap_size
    overlap_candidate = text[start_pos:]

    # Try to find a good starting point for overlap (sentence or phrase boundary)
    for separator in ['. ', '! ', '? ', '; ', ': ', ', ', ' - ', '-', ' (', ' ']:
        sep_pos = overlap_candidate.find(separator)
        if sep_pos != -1 and sep_pos > 0:
            return overlap_candidate[sep_pos + len(separator):]

    # If no good boundary found, use the full overlap
    return overlap_candidate

#### 4.3. Chunks text avoiding splitting words, while minimizing the number of chunks

In [146]:
def semantic_chunk_text(text, chunk_size, chunk_overlap):
    chunks = []
    remaining_text = text.strip()

    while remaining_text:
        if len(remaining_text) <= chunk_size:
            # Last chunk
            chunks.append(remaining_text)
            break

        # Find the best split point
        split_index = find_best_split_point(remaining_text, chunk_size)

        # Create current chunk
        current_chunk = remaining_text[:split_index].strip()

        if current_chunk:
            chunks.append(current_chunk)

        # Prepare next iteration with overlap
        remaining_text = remaining_text[split_index:].strip()

        if remaining_text and len(chunks) > 0:
            # Add overlap from previous chunk
            overlap = extract_overlap_text(current_chunk, chunk_overlap)
            if overlap and not remaining_text.startswith(overlap):
                remaining_text = overlap + " " + remaining_text

    return [chunk for chunk in chunks if chunk.strip()]

#### 4.2. Add contextual information to a chunk using extracted text summary

In [147]:
def contextualize_chunk(chunk, text_summary, llm):
    context = generate_ai_response(
        llm,
        SYSTEM_MESSAGE_CHUNK_CONTEXTUALIZER,
        HUMAN_MESSAGE_CHUNK_CONTEXTUALIZER,
        text_summary=text_summary,
        chunk_content=chunk
    )

    # Prepend context to the chunk content
    return f"{context.strip()} {chunk}"

#### 4.3. Main function to chunk PDF content with metadata

In [148]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document

def chunk_text(text, text_summary, llm, chunk_size, chunk_overlap):
    # Perform semantic chunking
    chunks = semantic_chunk_text(text, chunk_size, chunk_overlap)

    # Create chunks with metadata
    chunk_objects = []
    for i, chunk in enumerate(chunks):
        print(f"Processing chunk {i+1}/{len(chunks)}")

        contextualized_chunk = contextualize_chunk(chunk, text_summary, llm)

        chunk_objects.append(Document(
            page_content=contextualized_chunk,
            metadata={
                'id': f"chunk_{i+1}",
                'length': len(contextualized_chunk),
                'chunk_index': i,
                'total_chunks': len(chunks),
            }
        ))

    return chunk_objects

chunks = chunk_text(extracted_text, text_summary, llm, CHUNK_SIZE, CHUNK_OVERLAP)

Chunk
1 October 2024 edition A quick-start handbook for effective prompts 2 Writing effective prompts From the very beginning, Google Workspace was built to allow you to collaborate in real time with other people. Now, you can also collaborate with AI using Gemini for Google Workspace to help boost your productivity and creativity without sacrificing privacy or security. The embedded generative AI-powered features can help you write, organize information, create images, accelerate workflows, have richer meetings, and much more, all while using your favorite apps like Gmail, Google Docs, Google Drive, Google Sheets, Google Meet, Google Slides, and Gemini Advanced (the standalone chat experience available at gemini.google.com with enterprise- grade security). Gemini is accessible right where you are doing your work — with access to your personal knowledge base in Drive, Docs, Gmail, and more — so you can enhance and create powerful workflows across the Workspace apps with less tab switch

### 5. Initialize embedding model

In [149]:
from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(
     model=EMBEDDING_MODEL
  )

### 10. Initialize and populate Chroma DB

In [150]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory=None,
)

### 6. Initialize Conversation Buffer Memory

In [151]:
from langchain.memory import ConversationBufferMemory

memory = ConversationBufferMemory(
    memory_key="conversation_memory",
    return_messages=True,
    output_key="response"
)

/tmp/ipython-input-2148647518.py:3: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(


### 11. Define function to retrieve relevant content from database

In [152]:
def retrieve_relevant_context(
    vectorstore,
    query,
    k=4
):
    results = vectorstore.similarity_search(
        query,
        k=k
    )
    context = '\n'.join(
        result.page_content for result in results
    )

    return context.strip()

## Test Cases (Final Cell)

The final cell must contain your **test cases**.  
When executed, the AI should provide correct answers to the given questions **based on the PDF file**.


### AI Query Function

In this cell, you must implement the function **ask_ai(query)**.  
This function will be the final execution point of your pipeline (RAG / LLM).  


In [153]:
# ================================
# ❓ AI Query Function
# ================================

def ask_ai(query: str):
    """
    This function should execute your final RAG / LLM pipeline.
    Input:
        query (str): The question you want to ask the AI.
    Output:
        str: The AI's answer based on the PDF file.
    """
    # TODO: Implement your final execution logic here
    # Example steps:
    # 1. Retrieve relevant chunks
    # 2. Generate embeddings
    # 3. Call the model with your prompt + retrieved context
    # 4. Return the model's answer

    optimized_query = generate_ai_response(
      llm,
      SYSTEM_MESSAGE_QUERY_OPTIMIZER,
      HUMAN_MESSAGE_QUERY_OPTIMIZER,
      query=query,
      text_summary=text_summary,
    )

    context = retrieve_relevant_context(
      vectorstore,
      optimized_query,
      TOP_N_RESULTS,
    )

    ai_response = generate_ai_response(
      llm,
      SYSTEM_MESSAGE_RESPONDER,
      HUMAN_MESSAGE_RESPONDER,
      conversation_memory=memory.load_memory_variables({})['conversation_memory'],
      context=context,
      query=query,
      text_summary=text_summary,
    )

    memory.save_context({"input": query}, {"response": ai_response})


    return ai_response

### Test Queries

Use this cell to test your function with different queries.  
The answers must be generated correctly based on the PDF file.  


In [155]:
# ================================
# 🔍 Example Queries for Testing
# ================================

queries = [
    'My name is Bea',
    'What is my name',
    'What is pizza'
    # "How many words should effective prompts average?",
    # "List the four main areas for effective prompts.",
    # "What does 'persona' mean in prompt writing?",
    # "Name three business roles covered in this guide.",
    # "What is Gemini Advanced?",
]

# Call the AI with each query
for q in queries:
    print(f"Q: {q}")
    print(f"A: {ask_ai(q)}\n")


Q: My name is Bea
A: Hello, Bea! How can I assist you today?

Q: What is my name
A: Your name is Bea. How can I assist you today?

Q: What is pizza
A: I'm here to assist you with questions related to the PDF document. If you have any inquiries about effective prompt writing or Gemini AI features in Google Workspace, feel free to ask!

